---
**标题：** 上下文并行 (Context Parallelism, CP)

**类别：** context-parallelism

**难度：** 高级

**预计时间：** 45 分钟

---

## 概述

上下文并行（CP）让多个 GPU 共同分担长序列的注意力计算。本 notebook 将逐步构建这个概念：

1. 为什么长序列的注意力会撑爆内存
2. 将序列切分到多个 GPU
3. 环形注意力的技巧——旋转 KV 块
4. 在线 softmax 修正（保证数值精确）
5. 完整的工作模拟
6. Megatron-LM 的实际实现

### 前置知识
- 缩放点积注意力（Q, K, V）

> **Q、K、V 是什么？** 在注意力机制中，输入被投影为三个矩阵：**Query**（我在找什么？）、**Key**（我包含什么？）和 **Value**（我携带什么信息？）。注意力分数来自 Q 与 K 的匹配，然后用这些分数对 V 加权。
- [00-gpu-communication](../00-gpu-communication/) — P2P 和环形通信模式
- [04-sequence-parallelism](../04-sequence-parallelism/) — 序列并行基础

## 概念与原理

### 第一步：注意力与内存问题

从最小的例子开始。这是 **8 个 token**、**head_dim=4** 的注意力：

> **head_dim 是什么？** 每个注意力头操作的向量维度。比模型的完整隐藏维度小——例如，4096 维的模型有 32 个头，则 head_dim = 128。

In [ ]:
import torch
import numpy as np
from mp_tutorial.viz import draw_attention_heatmap

torch.manual_seed(0)
S, D = 8, 4  # 8个token，head_dim = 4
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)

# 计算注意力分数
scale = D ** -0.5
scores = (Q @ K.T) * scale
weights = torch.softmax(scores, dim=-1)

# 可视化：注意力权重矩阵是 S x S
token_labels = [f"t{i}" for i in range(S)]
draw_attention_heatmap(weights, title=f"Attention Weights ({S}x{S})", token_labels=token_labels)

这个 8x8 的矩阵很小。但注意力的内存随 **S x S** 增长——序列翻倍，内存翻四倍：

| 序列长度 | 分数矩阵大小 | 内存 (FP16) |
|---|---|---|
| 4,096 | 16M | 32 MB |
| 32,768 | 1B | 2 GB |
| 131,072 | 17B | 32 GB |

> **FP16 是什么？** 半精度浮点数——每个数字用 2 字节而非 4 字节（FP32）。训练大模型时的标准做法，用于节省内存。

在 128K token 时，仅分数矩阵就超出了大多数 GPU 的容量。**上下文并行解决这个问题。**

### 第二步：将序列切分到多个 GPU

思路：给每个 GPU 一**块**序列。2 个 GPU、8 个 token，每个 GPU 分到 4 个 token。

> **Chunk（块）是什么？** 序列的一段连续切片。如果有 8 个 token 和 2 个 GPU，token 0-3 是块 0，token 4-7 是块 1。

In [ ]:
from mp_tutorial.viz import draw_tensor_blocks

num_gpus = 2
chunk_size = S // num_gpus

# 切分 Q, K, V
q_chunks = list(Q.split(chunk_size, dim=0))
k_chunks = list(K.split(chunk_size, dim=0))
v_chunks = list(V.split(chunk_size, dim=0))

draw_tensor_blocks(q_chunks, title=f"Q split across {num_gpus} GPUs ({chunk_size} tokens each)")
draw_tensor_blocks(k_chunks, title=f"K split across {num_gpus} GPUs")

In [ ]:
from mp_tutorial.viz import draw_context_partition

# 一个真实句子切分到2个GPU — each GPU gets half the context
tokens = ["The", "cat", "sat", "on", "the", "mat", "and", "slept"]
draw_context_partition(tokens, num_gpus=2, q_chunks=q_chunks,
                        title="Context Parallelism: 8 tokens split across 2 GPUs")

In [ ]:
# 同一句子，现在用4个GPU — each gets just 2 tokens
q4 = list(Q.split(2, dim=0))
draw_context_partition(tokens, num_gpus=4, q_chunks=q4,
                        title="4-way CP: each GPU holds only 2 tokens of context")

每个 GPU 拥有自己的**本地 Q 块**。但要正确计算注意力，每个 Q 需要看到**所有的** K 和 V——不仅仅是本地的块。

完整的注意力矩阵有 4 个 tile。来看看：

In [ ]:
# 显示带有块边界的完整注意力矩阵
draw_attention_heatmap(
    weights,
    title="Full Attention — divided into tiles by GPU chunk",
    chunk_boundaries=[chunk_size],
    token_labels=token_labels
)

红线标出了块的边界。每个 GPU 需要计算**自己那行的 tile**（每个 KV 块对应一个 tile）。

> **Tile 是什么？** 完整注意力矩阵的一个子块。2 个 GPU 时，S×S 矩阵分成 2×2 的 tile 网格，每个 tile 大小为 (S/2)×(S/2)。

- GPU 0 需要：tile(Q0, K0) 和 tile(Q0, K1)
- GPU 1 需要：tile(Q1, K0) 和 tile(Q1, K1)

但每个 GPU 一开始只有自己的 KV 块。怎么获取另一个？

### 第三步：环形注意力——旋转 KV 块

与其把所有 KV 全部聚合到每个 GPU（代价太高！），我们**让 KV 块在环上传递**：

> **P2P（点对点）是什么？** GPU 之间的直接通信——GPU 0 直接把数据发给 GPU 1，不经过中心协调器。比广播给所有人更快。

In [ ]:
from mp_tutorial.viz import draw_ring_step_dataflow

# 第0步：每个GPU使用自己的本地KV
local_k = list(k_chunks)  # GPU 0 has K0, GPU 1 has K1
local_v = list(v_chunks)

step0_scores = []
step0_outputs = []
for gpu in range(num_gpus):
    s = (q_chunks[gpu] @ local_k[gpu].T) * scale
    w = torch.softmax(s, dim=-1)
    step0_scores.append(s)
    step0_outputs.append(w @ local_v[gpu])

draw_ring_step_dataflow(
    q_chunks, local_k, local_v, step0_scores, step0_outputs,
    step=0, num_gpus=num_gpus,
    title="Ring Step 0: Each GPU attends to its LOCAL KV"
)

In [ ]:
from mp_tutorial.distributed import simulate_p2p_kv_exchange

# KV旋转一步: GPU 0 gets K1, GPU 1 gets K0
local_k = simulate_p2p_kv_exchange(local_k)
local_v = simulate_p2p_kv_exchange(local_v)

step1_scores = []
step1_outputs = []
for gpu in range(num_gpus):
    s = (q_chunks[gpu] @ local_k[gpu].T) * scale
    w = torch.softmax(s, dim=-1)
    step1_scores.append(s)
    step1_outputs.append(w @ local_v[gpu])

draw_ring_step_dataflow(
    q_chunks, local_k, local_v, step1_scores, step1_outputs,
    step=1, num_gpus=num_gpus,
    title="Ring Step 1: KV blocks rotated — each GPU sees the OTHER chunk"
)

经过 2 步（= GPU 数量），每个 GPU 都已经看过了所有 KV 块！没有任何 GPU 持有过完整序列。

但有个问题——我们不能简单地平均两个部分输出。我们需要**在线 softmax 修正**。

### 第四步：在线 Softmax 修正

问题：对完整 key 维度做 softmax 需要一次看到所有分数。但我们是一个 tile 一个 tile 地计算的。

解决方案：维护一个滚动的 max 和 exp 之和，当新的 tile 到来时**重新缩放**累积的输出。看一个具体的例子：

> **在线算法是什么？** 一种逐块处理数据并维护滚动结果的算法，无需一次拿到所有数据。这里我们逐 tile 累积注意力结果，无需存储完整的 S×S 矩阵。

In [ ]:
# 逐步演示GPU 0的在线softmax修正
q0 = q_chunks[0]  # (4, 4)
k0_orig = k_chunks[0]  # K from chunk 0
k1_orig = k_chunks[1]  # K from chunk 1

print("=== GPU 0: Online Softmax Correction ===\n")

# --- Tile 0: Q0 @ K0 ---
scores_t0 = (q0 @ k0_orig.T) * scale
m0 = scores_t0.max(dim=-1, keepdim=True).values
exp0 = torch.exp(scores_t0 - m0)
l0 = exp0.sum(dim=-1, keepdim=True)
o0 = exp0 @ v_chunks[0]

print("Tile 0 (Q0 x K0):")
print(f"  scores shape: {scores_t0.shape}")
print(f"  row max (m):  {m0.squeeze().tolist()}")
print(f"  sum(exp):     {l0.squeeze().tolist()}")

# --- Tile 1: Q0 @ K1 ---
scores_t1 = (q0 @ k1_orig.T) * scale
m1 = scores_t1.max(dim=-1, keepdim=True).values
exp1 = torch.exp(scores_t1 - m1)
l1 = exp1.sum(dim=-1, keepdim=True)
o1 = exp1 @ v_chunks[1]

print(f"\nTile 1 (Q0 x K1):")
print(f"  row max (m):  {m1.squeeze().tolist()}")
print(f"  sum(exp):     {l1.squeeze().tolist()}")

# --- Merge: online correction ---
m_new = torch.maximum(m0, m1)
corr0 = torch.exp(m0 - m_new)  # rescale factor for tile 0
corr1 = torch.exp(m1 - m_new)  # rescale factor for tile 1
l_merged = corr0 * l0 + corr1 * l1
o_merged = (corr0 * o0 + corr1 * o1) / l_merged

print(f"\nMerged:")
print(f"  new max:        {m_new.squeeze().tolist()}")
print(f"  correction t0:  {corr0.squeeze().tolist()}")
print(f"  correction t1:  {corr1.squeeze().tolist()}")

# Verify against standard attention for GPU 0's rows
ref = (torch.softmax((q0 @ torch.cat([k0_orig, k1_orig]).T) * scale, dim=-1)
       @ torch.cat([v_chunks[0], v_chunks[1]]))
print(f"\nMax error vs standard attention: {(o_merged - ref).abs().max():.1e}")

修正因子 `exp(m_old - m_new)` 在新 tile 有更大的 max 时重新缩放之前的累积值。这和 FlashAttention 用的是完全相同的技巧——环形注意力只是把它扩展到了分布式场景。

**每个 tile 的更新规则：**

$$m_{\text{new}} = \max(m, m_k), \quad \ell_{\text{new}} = e^{m - m_{\text{new}}} \ell + e^{m_k - m_{\text{new}}} \ell_k, \quad O_{\text{new}} = e^{m - m_{\text{new}}} O + e^{m_k - m_{\text{new}}} O_k$$

## 可视化图解

### 完整的环形注意力流程（4 个 GPU）

现在扩展到 4 个 GPU，观看完整的环形传递：

In [ ]:
from mp_tutorial.viz import draw_ring_attention_steps

draw_ring_attention_steps(num_gpus=4, num_steps=4, title="Ring Attention: 4 GPUs, 4 Steps")

### 内存缩放

收益：有 N 个 GPU 时，每个 GPU 每次只需存放一个 (S/N) x (S/N) 的 tile。

In [ ]:
from mp_tutorial.viz import draw_cp_memory_scaling

draw_cp_memory_scaling(seq_lengths=[4096, 32768, 131072], max_gpus=8)

## 在大语言模型中的应用

| 并行维度 | 切分什么 | 典型规模 |
|---|---|---|
| **DP** | 批次 | 64-1024x |
| **TP** | 权重矩阵 / 注意力头 | 2-8x（节点内） |
| **SP** | LayerNorm、Dropout 激活值 | 与 TP 相同 |
| **CP** | 注意力 Q/K/V 沿序列维度 | 2-16x |
| **PP** | 层 | 4-16x |


> **NVLink 是什么？** 同一台机器（节点）内 GPU 之间的高速直连。比走网络快得多，这就是为什么 TP 通常仅限于 NVLink 连接的 GPU。

CP 与 TP/SP 正交——TP 切分**头**，CP 切分**序列**。它们可以组合使用。

**使用方：** Megatron-LM（`--context-parallel-size`），Llama 3.1（128K 上下文训练），DeepSpeed Ulysses（all-to-all 变体）。

## 动手实践

### 完整的环形注意力模拟

运行完整算法，验证结果与标准注意力一致：

In [ ]:
from mp_tutorial.distributed import check_gpu_env
check_gpu_env()

In [ ]:
from mp_tutorial.distributed import simulate_ring_attention

torch.manual_seed(42)
S, D, N = 32, 16, 4
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)

# 环形注意力
ring_out = simulate_ring_attention(Q, K, V, num_gpus=N, verbose=True)

# 标准注意力
ref = torch.softmax((Q @ K.T) * (D ** -0.5), dim=-1) @ V

print(f"\nMax error: {(ring_out - ref).abs().max():.1e}")
print(f"Match: {torch.allclose(ring_out, ref, atol=1e-5)}")

### 扩展测试

更多 GPU = 相同结果，只是切分方式不同：

In [ ]:
from mp_tutorial.formatting import comparison_table

torch.manual_seed(123)
S, D = 64, 32
Q = torch.randn(S, D)
K = torch.randn(S, D)
V = torch.randn(S, D)
ref = torch.softmax((Q @ K.T) * (D ** -0.5), dim=-1) @ V

rows = []
for n_gpus in [2, 4, 8]:
    out = simulate_ring_attention(Q, K, V, num_gpus=n_gpus)
    err = (out - ref).abs().max().item()
    rows.append([str(n_gpus), str(S // n_gpus), f"{err:.1e}",
                 "yes" if torch.allclose(out, ref, atol=1e-5) else "no"])

comparison_table(
    ["GPUs", "Chunk size", "Max error", "Match"],
    rows,
    title="Ring Attention correctness across GPU counts"
)

> **需要 GPU** — 在多卡 GPU 机器上运行以下 cell（推荐 4+ GPU）。

In [ ]:
# [GPU-REQUIRED]
# 真实分布式环形注意力 with torch.distributed
# 启动方式：torchrun --nproc_per_node=4 this_script.py

import torch
import torch.distributed as dist

def ring_attention_distributed(q_local, k_local, v_local, group=None):
    """使用真实P2P通信的环形注意力。"""
    rank = dist.get_rank(group)
    world_size = dist.get_world_size(group)
    chunk_size, head_dim = q_local.shape
    scale = head_dim ** -0.5

    m = torch.full((chunk_size, 1), float("-inf"), device=q_local.device)
    l = torch.zeros(chunk_size, 1, device=q_local.device)
    o = torch.zeros_like(q_local)

    k_recv = torch.empty_like(k_local)
    v_recv = torch.empty_like(v_local)
    k_curr, v_curr = k_local, v_local

    for step in range(world_size):
        # 在计算前启动异步P2P（通信与计算重叠）
        if step < world_size - 1:
            send_rank = (rank + 1) % world_size
            recv_rank = (rank - 1) % world_size
            send_k = dist.isend(k_curr, dst=send_rank, group=group)
            send_v = dist.isend(v_curr, dst=send_rank, group=group)
            recv_k = dist.irecv(k_recv, src=recv_rank, group=group)
            recv_v = dist.irecv(v_recv, src=recv_rank, group=group)

        # 计算注意力tile
        scores = (q_local @ k_curr.T) * scale
        tile_max = scores.max(dim=-1, keepdim=True).values
        tile_exp = torch.exp(scores - tile_max)
        tile_sum = tile_exp.sum(dim=-1, keepdim=True)
        tile_out = tile_exp @ v_curr

        # 在线softmax修正
        m_new = torch.maximum(m, tile_max)
        l = torch.exp(m - m_new) * l + torch.exp(tile_max - m_new) * tile_sum
        o = torch.exp(m - m_new) * o + torch.exp(tile_max - m_new) * tile_out
        m = m_new

        if step < world_size - 1:
            send_k.wait(); send_v.wait()
            recv_k.wait(); recv_v.wait()
            k_curr, v_curr = k_recv.clone(), v_recv.clone()

    return o / l

# dist.init_process_group("nccl")
# rank = dist.get_rank()
# ... 切分Q, K, V并调用ring_attention_distributed() ...

## Megatron 参考

Megatron-LM 在 `megatron/core/transformer/dot_product_attention.py` 中实现了 CP。

In [ ]:
from mp_tutorial.formatting import code_reference

code_reference(
    code="""# Megatron环形注意力的关键设计选择：
#
# 1. 通信-计算重叠
#    - KV的isend/irecv在计算前启动
#    - GPU计算注意力tile的同时网络传输下一个KV块
#
# 2. 因果掩码优化
#    - 对于自回归模型，GPU i跳过来自
#      位置 > i 的KV块（未来token无法注意到）
#    - 对因果模型可节省约50%的环形步骤
#
# 3. CP + TP 集成
#    - CP组：切分序列维度
#    - TP组：切分注意力头
#    - 正交 — 两者在同一前向传播中同时活动
#    - 总GPU数 = DP x TP x PP x CP""",
    source="Megatron-LM",
    filepath="megatron/core/transformer/dot_product_attention.py"
)

## 总结与延伸阅读

**核心要点：**
- CP 将注意力分布到多个 GPU：每 GPU 内存从 O(S^2) 降至 O(S^2/N)
- 环形注意力在环上旋转 KV 块——每个 GPU 同一时刻只持有一个块
- 在线 softmax 修正使分块注意力在数值上精确（与 FlashAttention 相同的技巧）
- CP 与 TP/SP 正交——TP 切头，CP 切序列，可以干净地组合
- 通信与计算可以重叠；因果掩码优化可跳过约 50% 的环形步骤

> **因果掩码是什么？** 在自回归（从左到右）模型中，token i 只能关注 token 0..i，不能看到未来的 token。因果掩码通过屏蔽序列中后方位置的注意力来实现这一点。

**延伸阅读：**
- [Ring Attention with Blockwise Transformers](https://arxiv.org/abs/2310.01889) — Liu et al., 2023
- [FlashAttention-2](https://arxiv.org/abs/2307.08691) — Dao, 2023（在线 softmax 修正的来源）
- [Megatron-LM](https://github.com/NVIDIA/Megatron-LM) — NVIDIA 分布式训练框架
- [DeepSpeed Ulysses](https://arxiv.org/abs/2309.14509) — all-to-all CP 变体